In [23]:
%pip install numpy pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [24]:
import json
import re
import numpy as np
import pandas as pd

What variable we want to explore the correlation with single or multi listing

estimated_revenue_l365d
estimated_occupancy_l365d
number_of_reviews_ly
price
...

In [25]:
listings_raw = pd.read_csv('listings_raw.csv')
reviews_raw = pd.read_csv('reviews_raw.csv')

listings_raw.head(5)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20260616211424,2026-06-16,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://a0.muscache.com/pictures/44627226/0e72...,4621559,...,4.56,3.22,3.67,NaN,NaN,1,1,0,0,0.06
1,3820211,https://www.airbnb.com/rooms/3820211,20260616211424,2026-06-16,city scrape,Restored Precinct in Center Sq. w/Parking,Step into the charming and comfy 1BR/1BA apart...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,19648678,...,4.81,4.82,4.78,NaN,NaN,7,7,0,0,2.20
2,5651579,https://www.airbnb.com/rooms/5651579,20260616211424,2026-06-16,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",NaN,https://a0.muscache.com/pictures/b3fc42f3-6e5e...,29288920,...,4.87,4.75,4.63,NaN,NaN,2,1,1,0,3.00
3,6623339,https://www.airbnb.com/rooms/6623339,20260616211424,2026-06-16,city scrape,Center Sq. Loft in Converted Precinct w/ Parking,Step into the charming and comfy 1BR/1BA apart...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,19648678,...,4.70,4.80,4.72,NaN,NaN,7,7,0,0,2.47
4,9501054,https://www.airbnb.com/rooms/9501054,20260616211424,2026-06-16,city scrape,Spacious suite with full bath by Capital Center,Great location within walking distance to the ...,NaN,https://a0.muscache.com/pictures/45153167-d704...,29288920,...,4.87,4.77,4.68,NaN,NaN,2,1,1,0,3.71


In [26]:
listings_raw

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,2992450,https://www.airbnb.com/rooms/2992450,20260616211424,2026-06-16,city scrape,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,NaN,https://a0.muscache.com/pictures/44627226/0e72...,4621559,...,4.56,3.22,3.67,NaN,NaN,1,1,0,0,0.06
1,3820211,https://www.airbnb.com/rooms/3820211,20260616211424,2026-06-16,city scrape,Restored Precinct in Center Sq. w/Parking,Step into the charming and comfy 1BR/1BA apart...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,19648678,...,4.81,4.82,4.78,NaN,NaN,7,7,0,0,2.20
2,5651579,https://www.airbnb.com/rooms/5651579,20260616211424,2026-06-16,city scrape,Large studio apt by Capital Center & ESP@,"Spacious studio with hardwood floors, fully eq...",NaN,https://a0.muscache.com/pictures/b3fc42f3-6e5e...,29288920,...,4.87,4.75,4.63,NaN,NaN,2,1,1,0,3.00
3,6623339,https://www.airbnb.com/rooms/6623339,20260616211424,2026-06-16,city scrape,Center Sq. Loft in Converted Precinct w/ Parking,Step into the charming and comfy 1BR/1BA apart...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,19648678,...,4.70,4.80,4.72,NaN,NaN,7,7,0,0,2.47
4,9501054,https://www.airbnb.com/rooms/9501054,20260616211424,2026-06-16,city scrape,Spacious suite with full bath by Capital Center,Great location within walking distance to the ...,NaN,https://a0.muscache.com/pictures/45153167-d704...,29288920,...,4.87,4.77,4.68,NaN,NaN,2,1,1,0,3.71
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,1681915932025735153,https://www.airbnb.com/rooms/1681915932025735153,20260616211424,2026-06-16,city scrape,Perfect Albany Getaway Near Hospitals & Downtown,Stylish downtown Albany apartment unit near be...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,512736975,...,NaN,NaN,NaN,NaN,NaN,2,2,0,0,NaN
486,1689544536474585971,https://www.airbnb.com/rooms/1689544536474585971,20260616211424,2026-06-16,city scrape,Great spot in Albany,"This special place is close to everything, mak...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,3073250,...,NaN,NaN,NaN,NaN,NaN,1,1,0,0,NaN
487,1697861560802480129,https://www.airbnb.com/rooms/1697861560802480129,20260616211424,2026-06-16,city scrape,AMC Cozy Bedroom,Located in the heart of Albany. The room offer...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,522436003,...,NaN,NaN,NaN,NaN,NaN,6,0,6,0,NaN
488,1697894513901184937,https://www.airbnb.com/rooms/1697894513901184937,20260616211424,2026-06-16,city scrape,MC Washington Park,Located in the heart of Albany. The room offer...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,522436003,...,NaN,NaN,NaN,NaN,NaN,6,0,6,0,NaN


clean price into int for comparison

In [27]:
def parse_price(df):
    df = df.copy()
    
    def parse_price_int(price):
        """
        Independently re-implement the parsing logic in a different way 
        """
        if pd.isna(price):
            return np.nan

        clean_price = str(price).replace("$", "").replace(",", "").strip()
        try:
            return float(clean_price)
        except:
            return np.nan

    df['price_int'] = df['price'].apply(parse_price_int)

    return df

turn the number of listings into only single or multilisting

In [28]:
def listing_class(df):
    df = df.copy()
    
    def listing_class_logic(num_listings):

        if pd.isna(num_listings):
            return np.nan
        else:
            num_listings = int(num_listings)
        
        if num_listings <= 0:
            return np.nan
        elif num_listings == 1:
            return "Singlelisting"
        else:
            return "Multilisting"

    df['listing_class'] = df['calculated_host_listings_count'].apply(listing_class_logic)

    return df

# EDA

## Before Preprocessing

In [29]:
def column_audit(df):
    rows = len(df)

    result = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_missing": df.notna().sum(),
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean().mul(100),
        "unique_count": df.nunique(dropna=True)
    })

    result["unique_percent"] = (
        result["unique_count"] / rows * 100
    )

    return result.sort_values(
        ["missing_percent", "unique_count"],
        ascending=[False, True]
    )

In [30]:
listings_audit = column_audit(listings_raw)
reviews_audit = column_audit(reviews_raw)

display(listings_audit)
display(reviews_audit)

,dtype,non_missing,missing_count,missing_percent,unique_count,unique_percent
neighborhood_overview,float64,0,490,100.0,0,0.000000
host_since,float64,0,490,100.0,0,0.000000
host_response_time,float64,0,490,100.0,0,0.000000
host_response_rate,float64,0,490,100.0,0,0.000000
host_acceptance_rate,float64,0,490,100.0,0,0.000000
...,...,...,...,...,...,...
amenities,object,490,0,0.0,474,96.734694
picture_url,object,490,0,0.0,482,98.367347
name,object,490,0,0.0,489,99.795918
id,int64,490,0,0.0,490,100.000000


,dtype,non_missing,missing_count,missing_percent,unique_count,unique_percent
comments,object,29038,8,0.027543,28100,96.743097
listing_id,int64,29046,0,0.000000,430,1.480410
date,object,29046,0,0.000000,3579,12.321834
reviewer_name,object,29046,0,0.000000,7361,25.342560
reviewer_id,int64,29046,0,0.000000,23896,82.269504
id,int64,29046,0,0.000000,29046,100.000000


In [31]:
listings_all_missing =  listings_raw.columns[
    listings_raw.isna().all()
].tolist()

reviews_all_missing = reviews_raw.columns[
    reviews_raw.isna().all()
].tolist()

print("Listings columns with 100% missing:")
print(listings_all_missing)

print("Reviews columns with 100% missing:")
print(reviews_all_missing)

Listings columns with 100% missing:
['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'calendar_updated', 'license', 'instant_bookable']
Reviews columns with 100% missing:
[]


In [32]:
listings_work = listings_raw.drop(
    columns=listings_all_missing
).copy()

reviews_work = reviews_raw.copy()

print("Listings before:", listings_raw.shape)
print("Listings after:", listings_work.shape)

print("Reviews before:", reviews_raw.shape)
print("Reviews after:", reviews_work.shape)

Listings before: (490, 90)
Listings after: (490, 76)
Reviews before: (29046, 6)
Reviews after: (29046, 6)


In [35]:
listings = pd.read_csv('listings.csv')
reviews= pd.read_csv('reviews.csv')


In [36]:
print("Listings :", listings.shape)
print("Reviews :", reviews.shape)

Listings : (25728, 90)
Reviews : (1026690, 6)
